In [ ]:
############################################################
# SMART RESUME SCREENING
# Random Forest + SVM
############################################################

############################################################
# 1. IMPORT LIBRARIES
############################################################

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:

############################################################
# 2. LOAD DATA
############################################################

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

print(train.head())
print(train.shape)


In [ ]:

############################################################
# 3. TEXT PREPROCESSING (BEST VERSION)
############################################################

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower()

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)


train["clean_text"] = train["Text"].apply(clean_text)
test["clean_text"] = test["Text"].apply(clean_text)


In [ ]:

############################################################
# 4. LABEL ENCODING
############################################################

label_encoder = LabelEncoder()

train["label_encoded"] = label_encoder.fit_transform(train["Label"])


In [ ]:

############################################################
# 5. TRAIN TEST SPLIT
############################################################

X_train, X_val, y_train, y_val = train_test_split(
    train["clean_text"],
    train["label_encoded"],
    test_size=0.2,
    random_state=42
)


In [ ]:

############################################################
# 6. TF-IDF FEATURE EXTRACTION
############################################################

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)


In [ ]:

############################################################
# 7. RANDOM FOREST MODEL
############################################################

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)

rf_pred = rf_model.predict(X_val_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_val, rf_pred))

print(classification_report(y_val, rf_pred))



In [ ]:

############################################################
# 8. SVM MODEL
############################################################

svm_model = SVC(
    kernel='linear',
    C=1,
    probability=True
)

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_val_tfidf)

print("SVM Accuracy:", accuracy_score(y_val, svm_pred))

print(classification_report(y_val, svm_pred))


In [ ]:


############################################################
# 9. TEST PREDICTION
############################################################

X_test = tfidf.transform(test["clean_text"])

rf_test_pred = rf_model.predict(X_test)
svm_test_pred = svm_model.predict(X_test)

test["RF_Prediction"] = label_encoder.inverse_transform(rf_test_pred)
test["SVM_Prediction"] = label_encoder.inverse_transform(svm_test_pred)

print(test.head())



In [ ]:

############################################################
# 10. SAVE RESULTS
############################################################

test.to_csv("resume_predictions.csv", index=False)

print("Predictions Saved")